# Week 4 — Theory
## MLOps experiment tracking and live metric monitoring

Once your training loop is bigger than a notebook, you need a system that records every
run, lets you compare across runs, and points back to the artefacts each run produced.
That system is an **experiment tracker**. The two we cover are **MLflow** (open source,
local-first) and **Weights & Biases** (managed service with a generous free tier).

This notebook is the conceptual half of the week. It covers:

1. What an experiment tracker stores, and the schema underneath the UI.
2. What to **log** during training — and what not to log.
3. **Sweep design**: grid vs. random vs. Bayesian, when to use which.
4. The four sweep-analysis plots: **parallel coordinates**, **parameter importance**,
   **scatter of metric vs. hyperparameter**, and **slice plots**.
5. Artefact versioning — making a chart in a paper reproducible from a run id.


## 1. What an experiment tracker stores

Both MLflow and W&B converge on roughly the same schema:

```
experiment
└── run                                  # one row per training run
    ├── params              {k: v}       # fixed at the start (hyperparameters)
    ├── metrics            [{step, k: v}, ...]   # time-series of scalars
    ├── tags                {k: v}       # categorical metadata (git sha, dataset...)
    └── artefacts           [files...]   # checkpoints, figures, configs, data
```

**Params** are the inputs you control. **Metrics** are the outputs you measure.
**Tags** are the metadata you need to compare and filter runs. **Artefacts** are
everything else — model files, generated figures, profiler traces.

The whole system is designed for **filter, group, plot** workflows:

- *"Show me all runs from this branch where `lr > 1e-3` and group by `optimizer`."*
- *"Plot val/loss over time for every run of this sweep, coloured by `weight_decay`."*
- *"For my best run, give me the validation predictions artefact."*

This is the spreadsheet you wish you had been keeping by hand all along.

## 2. What to log

A short, opinionated list.

**Always log:**

- Every hyperparameter you passed to the training loop. This includes the trivial ones
  (`batch_size`, `seed`) that you swear you'll remember.
- Train loss and validation loss at every epoch (and every $N$ steps if your epoch is
  long).
- At least one validation metric that is **not** the loss — accuracy, F1, BLEU,
  whatever your downstream task uses.
- The **best checkpoint** as an artefact, with a key like `best_val_loss`.
- A **summary** at the end of training: best val score, total steps, wall-clock seconds.
- The exact **git commit** the run was launched from, and the contents of your
  `requirements.txt` or `environment.yml`.

**Often forgotten but high-value:**

- **System metrics** — GPU utilization, memory, throughput. A run that finishes at
  10 % GPU utilization is wasting your compute budget. Both trackers can capture this
  automatically.
- **Gradient norms** and **parameter histograms** every so often. Cheap, and the cause
  of "my loss diverged at step 8000" is almost always visible here in retrospect.
- **A few qualitative samples** — model predictions on a fixed validation set, every
  $N$ epochs. Lets you watch the model learn, not just watch the loss curve.

**What not to log:**

- **The entire dataset.** Log the *path* and the *hash* of the dataset; not the bytes.
- **Per-step learning rate as a scalar** when you have a known schedule — log the
  schedule's parameters and reconstruct it. Saves bandwidth and clutter.
- **Anything sensitive.** API keys, raw user data, anything that would embarrass you in
  a security review.

## 3. Sweep design

A **sweep** runs your training script repeatedly with different hyperparameter values.
There are three families.

### Grid search

Pick a set of values for each hyperparameter; train on the Cartesian product. Cost grows
multiplicatively. **Use when you have ≤ 3 hyperparameters and ≤ 5 values each, and you
want a complete picture.**

### Random search

Sample each hyperparameter independently from a prior. Counter-intuitive but proven:
random search **usually beats grid search at the same compute budget**, because grid
search wastes evaluations on values of unimportant hyperparameters
(Bergstra & Bengio, 2012). **Use as your default for any sweep with 4+ hyperparameters.**

### Bayesian / TPE / multi-armed bandit

Use the results of earlier runs to choose the next configuration. Tools: **Optuna** (TPE),
W&B Sweeps (Bayesian), Ray Tune (many). **Use when each run is expensive and the
hyperparameter space is non-trivial.** The cost is added system complexity and a worse
worst case (the algorithm can over-commit to a local optimum).

### A practical default

1. Start with a **random search** of 20–30 runs over a wide hyperparameter range.
2. Inspect the parallel-coordinates plot. Identify the 2–3 hyperparameters that actually
   matter for your metric.
3. Run a **smaller, denser random or Bayesian search** in the region those parameters
   suggested. The other hyperparameters can be fixed at the best values from step 1.

This is the **explore then exploit** pattern, and it consistently beats either extreme
on real-world projects.

## 4. The four sweep-analysis plots

### Parallel coordinates

One vertical axis per hyperparameter, one extra axis for the metric. Each run is a line
that connects its values across the axes. Colour by metric value.

**What you can read off it:**

- Which hyperparameter values are associated with high or low metric.
- Interactions — two parameters that are individually fine but bad together stand out
  as crossed lines.

**What it does not tell you:** which parameter is *causally* most important. For that
you need a separate analysis.

### Parameter importance

Train a regression on (hyperparams → metric) across runs and read off the feature
importance (or the partial dependence) of each hyperparameter. W&B has this built in
and Optuna ships `optuna.importance.get_param_importances`.

**Caveat.** Importance estimated from a random search of 20 runs is itself uncertain;
treat the top-3 as suggestive, not as a ranking with 95 % CI.

### Metric vs. one hyperparameter

The simplest possible diagnostic. For each hyperparameter independently, scatter the
metric against the hyperparameter value (one point per run). Useful for picking out
clear monotonic relationships.

### Slice plots

A row of "metric vs. hyperparameter" subplots, one per hyperparameter. Same information,
nicer layout for a paper figure. Plotly's `optuna.visualization.plot_slice` produces
these in two lines.

## 5. Artefact versioning

The promise of an experiment tracker is **reproducibility from a run id**. To deliver on
this, every artefact a run produces — checkpoints, generated figures, validation
predictions, profiler traces — needs to be logged with the run.

A useful pattern:

```python
# At the end of training
mlflow.log_artifact("checkpoints/best.pt")
mlflow.log_artifact("figures/val_predictions.png")
mlflow.log_dict({"git_sha": SHA, "config": cfg}, "config.json")
```

And on reload:

```python
client = mlflow.MlflowClient()
run = client.get_run(run_id)
ckpt = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path="best.pt")
model = MyModel.load_from(ckpt)
```

You should be able to take any chart in your paper, find the run id in the caption (or
in a supplementary file), and re-render the chart from the artefacts on disk. **If you
can't, the chart is not really reproducible.**

## Summary

- An experiment tracker is the spreadsheet you would have built by hand. Use one.
- **Log generously** at training time; storage is cheap, regret is expensive.
- Default sweep strategy: **random search, then exploit**. Use Bayesian methods when
  runs are expensive.
- Build a **parallel-coordinates plot** for every sweep — it answers most "which
  hyperparameter matters?" questions.
- **Tie artefacts to run ids**. A paper chart that cannot be re-rendered from its run
  id is not reproducible.

In the lab, we instrument a PyTorch training loop with MLflow, run a small sweep with
Optuna, and reproduce the standard sweep-analysis plots both inside the trackers and
offline in Matplotlib.
